# PyROOT TH2 response fit and file comparison

这个 notebook 直接使用原生 PyROOT 从 ROOT 文件里的 response / cross-talk `TH2` 对象做线性拟合，不读取 `events` tree，也不依赖 `uproot`。核心流程是：

1. 对每个输入 ROOT 文件读取四张二维图：`v2_wrt_psi2_vs_eps2`、`v3_wrt_psi3_vs_eps3`、`v2_wrt_psi2_vs_eps3`、`v3_wrt_psi3_vs_eps2`。
2. 用 `TH2::ProfileX` 得到每个 `epsilon` bin 上的平均 `v_{n|Psi_n}`。
3. 在 `TProfile` 上用 `TF1` 做两种 ROOT 原生拟合：自由截距 `p0 + k*x` 和过原点 `k*x`。
4. 按物理分组对比不同文件：`v_n / epsilon_n` 放在一张 canvas，`v_n / epsilon_m` 放在另一张 canvas。

如果要比较多个结果文件，在下面的 `INPUT_FILES` 中增加 `(path, label)` 即可。label 会进入图例和输出表。


In [1]:
from pathlib import Path
import csv
import math
import re

import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptFit(0)


## 输入与拟合配置

`FIT_X_RANGE = None` 表示使用 `TH2` 的完整 x 轴存储范围。若只想拟合 response-test 常用密集区，可设为 `(0.0, 0.35)`。显示窗口默认保持在 response-test 常用范围，但不改变实际拟合范围。


In [13]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "qa").exists() and (PROJECT_ROOT.parent / "qa").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_FILES = [
    (PROJECT_ROOT / "qa" / "test_b8_response_023.root", "response_023"),
    (PROJECT_ROOT / "qa" / "test_b8_response_023_mix.root", "dense_mix"),
    (PROJECT_ROOT / "qa" / "test_b8_023_newrap.root", "newrap"),
]

OUTPUT_DIR = PROJECT_ROOT / "qa" / "vn_epsn_pyroot_th2_fit"
SAVE_OUTPUTS = True

# FIT_X_RANGE = None
FIT_X_RANGE = (0.0,0.35)
DISPLAY_X_RANGE = (0.0, 0.35)
DISPLAY_Y_RANGE = (-0.15, 0.15)
MIN_PROFILE_POINTS = 2
FIT_OPTIONS = "QRS0"


In [14]:
OBSERVABLE_GROUPS = {
    "vn_over_epsn": [
        {
            "key": "v2_over_eps2",
            "object": "v2_wrt_psi2_vs_eps2",
            "title": "v_{2|#Psi_{2}} vs #epsilon_{2}",
            "x_title": "#epsilon_{2}",
            "y_title": "v_{2|#Psi_{2}}",
        },
        {
            "key": "v3_over_eps3",
            "object": "v3_wrt_psi3_vs_eps3",
            "title": "v_{3|#Psi_{3}} vs #epsilon_{3}",
            "x_title": "#epsilon_{3}",
            "y_title": "v_{3|#Psi_{3}}",
        },
    ],
    "vn_over_epsm": [
        {
            "key": "v2_over_eps3",
            "object": "v2_wrt_psi2_vs_eps3",
            "title": "v_{2|#Psi_{2}} vs #epsilon_{3}",
            "x_title": "#epsilon_{3}",
            "y_title": "v_{2|#Psi_{2}}",
        },
        {
            "key": "v3_over_eps2",
            "object": "v3_wrt_psi3_vs_eps2",
            "title": "v_{3|#Psi_{3}} vs #epsilon_{2}",
            "x_title": "#epsilon_{2}",
            "y_title": "v_{3|#Psi_{3}}",
        },
    ],
}

GROUP_TITLES = {
    "vn_over_epsn": "same-harmonic response: v_{n} / #epsilon_{n}",
    "vn_over_epsm": "cross-harmonic mixing: v_{n} / #epsilon_{m}",
}

COLORS = [ROOT.kBlue + 1, ROOT.kRed + 1, ROOT.kGreen + 2, ROOT.kMagenta + 1, ROOT.kOrange + 7, ROOT.kCyan + 2]
MARKERS = [20, 21, 22, 23, 24, 25]


In [15]:
def root_safe_name(text):
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(text)).strip("_")


def normalise_input_files(entries):
    specs = []
    for entry in entries:
        if isinstance(entry, (str, Path)):
            path = Path(entry)
            label = path.stem
        else:
            path = Path(entry[0])
            label = str(entry[1]) if len(entry) > 1 else path.stem

        if not path.is_absolute():
            path = PROJECT_ROOT / path
        specs.append({"path": path.resolve(), "label": label})
    return specs


def load_th2(file_path, object_name):
    root_file = ROOT.TFile.Open(str(file_path), "READ")
    if not root_file or root_file.IsZombie():
        raise OSError(f"Failed to open ROOT file: {file_path}")

    try:
        obj = root_file.Get(object_name)
        if not obj:
            raise KeyError(f"Missing object {object_name!r} in {file_path}")
        if not obj.InheritsFrom("TH2"):
            raise TypeError(f"Object {object_name!r} in {file_path} is {obj.ClassName()}, not TH2")

        clone = obj.Clone(root_safe_name(f"{file_path.stem}_{object_name}_source"))
        clone.SetDirectory(0)
        return clone
    finally:
        root_file.Close()


def profile_nonempty_points(profile):
    n_points = 0
    for ix in range(1, profile.GetNbinsX() + 1):
        if profile.GetBinEntries(ix) > 0:
            n_points += 1
    return n_points


def fit_range_for(histogram):
    if FIT_X_RANGE is not None:
        return FIT_X_RANGE
    axis = histogram.GetXaxis()
    return axis.GetXmin(), axis.GetXmax()


def fit_profile(profile, expression, name, x_min, x_max):
    function = ROOT.TF1(name, expression, x_min, x_max)
    fit_result = profile.Fit(function, FIT_OPTIONS)
    try:
        status = int(fit_result)
    except TypeError:
        status = fit_result.Status()
    return function, status


In [16]:
def fit_one_observable(input_spec, group_name, observable, style_index):
    file_path = input_spec["path"]
    file_label = input_spec["label"]
    hist = load_th2(file_path, observable["object"])
    x_min, x_max = fit_range_for(hist)

    base_name = root_safe_name(f"{group_name}_{observable['key']}_{file_label}")
    profile = hist.ProfileX(f"{base_name}_profile", 1, hist.GetNbinsY(), "")
    profile.SetDirectory(0)
    profile.SetTitle(f"{observable['title']};{observable['x_title']};{observable['y_title']}")

    n_points = profile_nonempty_points(profile)
    if n_points < MIN_PROFILE_POINTS:
        raise RuntimeError(
            f"{file_label}:{observable['object']} has only {n_points} non-empty profile bins; "
            f"need at least {MIN_PROFILE_POINTS}"
        )

    free_fn, free_status = fit_profile(profile, "[0] + [1]*x", f"{base_name}_free", x_min, x_max)
    origin_fn, origin_status = fit_profile(profile, "[0]*x", f"{base_name}_origin", x_min, x_max)

    row = {
        "file_label": file_label,
        "file_path": str(file_path),
        "group": group_name,
        "observable": observable["key"],
        "root_object": observable["object"],
        "th2_entries": hist.GetEntries(),
        "profile_points": n_points,
        "fit_x_min": x_min,
        "fit_x_max": x_max,
        "free_status": free_status,
        "free_intercept": free_fn.GetParameter(0),
        "free_intercept_error": free_fn.GetParError(0),
        "free_slope": free_fn.GetParameter(1),
        "free_slope_error": free_fn.GetParError(1),
        "free_chi2": free_fn.GetChisquare(),
        "free_ndf": free_fn.GetNDF(),
        "free_prob": free_fn.GetProb() if free_fn.GetNDF() > 0 else math.nan,
        "origin_status": origin_status,
        "origin_slope": origin_fn.GetParameter(0),
        "origin_slope_error": origin_fn.GetParError(0),
        "origin_chi2": origin_fn.GetChisquare(),
        "origin_ndf": origin_fn.GetNDF(),
        "origin_prob": origin_fn.GetProb() if origin_fn.GetNDF() > 0 else math.nan,
    }

    return {
        "input": input_spec,
        "group": group_name,
        "observable": observable,
        "histogram": hist,
        "profile": profile,
        "free_fn": free_fn,
        "origin_fn": origin_fn,
        "row": row,
        "style_index": style_index,
    }


In [17]:
input_specs = normalise_input_files(INPUT_FILES)
fit_results = {group_name: {obs["key"]: [] for obs in observables} for group_name, observables in OBSERVABLE_GROUPS.items()}
summary_rows = []

for style_index, input_spec in enumerate(input_specs):
    for group_name, observables in OBSERVABLE_GROUPS.items():
        for observable in observables:
            try:
                result = fit_one_observable(input_spec, group_name, observable, style_index)
            except Exception as exc:
                print(f"[skip] {input_spec['label']} {observable['object']}: {exc}")
                continue

            fit_results[group_name][observable["key"]].append(result)
            summary_rows.append(result["row"])

print(f"Finished {len(summary_rows)} TH2 profile fits from {len(input_specs)} input file(s).")


Finished 12 TH2 profile fits from 3 input file(s).


In [18]:
try:
    import pandas as pd
except Exception:
    pd = None

if pd is not None:
    summary_table = pd.DataFrame(summary_rows)
    display(summary_table)
else:
    for row in summary_rows:
        print(row)


,file_label,file_path,group,observable,root_object,th2_entries,profile_points,fit_x_min,fit_x_max,free_status,...,free_slope_error,free_chi2,free_ndf,free_prob,origin_status,origin_slope,origin_slope_error,origin_chi2,origin_ndf,origin_prob
0,response_023,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v2_over_eps2,v2_wrt_psi2_vs_eps2,5000.0,29,0.0,0.35,0,...,1.124029e-09,1.890478e+15,27,0.000000e+00,0,0.009272,6.956047e-10,7.304170e+16,28,0.000000e+00
1,response_023,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v3_over_eps3,v3_wrt_psi3_vs_eps3,5000.0,25,0.0,0.35,1,...,2.621854e-08,4.019390e+03,23,0.000000e+00,0,0.207547,1.657378e-09,1.762676e+02,24,3.771493e-25
2,response_023,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsm,v2_over_eps3,v2_wrt_psi2_vs_eps3,5000.0,25,0.0,0.35,1,...,6.423648e-09,1.355269e+04,23,0.000000e+00,0,-0.056604,4.060644e-10,1.067106e+04,24,0.000000e+00
3,response_023,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsm,v3_over_eps2,v3_wrt_psi3_vs_eps2,5000.0,29,0.0,0.35,0,...,3.687622e-10,3.707107e+16,27,0.000000e+00,0,-0.034555,2.689719e-10,1.616361e+17,28,0.000000e+00
4,dense_mix,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v2_over_eps2,v2_wrt_psi2_vs_eps2,5000.0,29,0.0,0.35,0,...,1.566739e-09,1.103850e+16,27,0.000000e+00,0,-0.035386,1.016770e-09,4.907169e+16,28,0.000000e+00
5,dense_mix,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v3_over_eps3,v3_wrt_psi3_vs_eps3,5000.0,25,0.0,0.35,1,...,2.621854e-08,5.078190e+03,23,0.000000e+00,0,0.207547,1.657378e-09,3.711497e+02,24,6.092630e-64
6,dense_mix,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsm,v2_over_eps3,v2_wrt_psi2_vs_eps3,5000.0,25,0.0,0.35,0,...,8.222288e-09,8.561099e+05,23,0.000000e+00,0,-0.017902,3.730496e-10,5.823176e+16,24,0.000000e+00
7,dense_mix,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsm,v3_over_eps2,v3_wrt_psi3_vs_eps2,5000.0,29,0.0,0.35,0,...,4.125766e-10,1.430712e+15,27,0.000000e+00,0,-0.062853,3.033626e-10,3.018836e+17,28,0.000000e+00
8,newrap,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v2_over_eps2,v2_wrt_psi2_vs_eps2,5000.0,34,0.0,0.35,0,...,8.808129e-03,3.802302e+01,32,2.140460e-01,0,0.129647,2.547542e-03,4.683948e+01,33,5.587750e-02
9,newrap,/Users/allenzhou/Research_software/Blast_wave/...,vn_over_epsn,v3_over_eps3,v3_wrt_psi3_vs_eps3,5000.0,31,0.0,0.35,1,...,2.474395e-08,1.805830e+04,29,0.000000e+00,0,0.377049,1.564164e-09,3.422632e+03,30,0.000000e+00


In [19]:
def style_fit_objects(result):
    color = COLORS[result["style_index"] % len(COLORS)]
    marker = MARKERS[result["style_index"] % len(MARKERS)]
    profile = result["profile"]
    profile.SetMarkerColor(color)
    profile.SetLineColor(color)
    profile.SetMarkerStyle(marker)
    profile.SetMarkerSize(0.9)

    result["free_fn"].SetLineColor(color)
    result["free_fn"].SetLineWidth(2)
    result["free_fn"].SetLineStyle(1)
    result["origin_fn"].SetLineColor(color)
    result["origin_fn"].SetLineWidth(2)
    result["origin_fn"].SetLineStyle(2)


def apply_display_ranges(profile):
    if DISPLAY_X_RANGE is not None:
        profile.GetXaxis().SetRangeUser(*DISPLAY_X_RANGE)
    if DISPLAY_Y_RANGE is not None:
        profile.GetYaxis().SetRangeUser(*DISPLAY_Y_RANGE)


def draw_group_canvas(group_name):
    observables = OBSERVABLE_GROUPS[group_name]
    canvas = ROOT.TCanvas(root_safe_name(f"canvas_{group_name}"), GROUP_TITLES[group_name], 700, 400)
    canvas.Divide(len(observables), 1)
    legends = []

    for pad_index, observable in enumerate(observables, start=1):
        canvas.cd(pad_index)
        ROOT.gPad.SetGrid(1, 1)
        results = fit_results[group_name][observable["key"]]
        legend = ROOT.TLegend(0.14, 0.67, 0.88, 0.88)
        legend.SetBorderSize(0)
        legend.SetFillStyle(0)
        legends.append(legend)

        first = True
        for result in results:
            style_fit_objects(result)
            profile = result["profile"]
            profile.SetTitle(f"{observable['title']};{observable['x_title']};{observable['y_title']}")
            apply_display_ranges(profile)
            profile.Draw("E" if first else "E SAME")
            result["free_fn"].Draw("SAME")
            result["origin_fn"].Draw("SAME")

            row = result["row"]
            legend.AddEntry(
                profile,
                f"{row['file_label']}: k={row['free_slope']:.4g}, k0={row['origin_slope']:.4g}",
                "lep",
            )
            first = False

        legend.AddEntry(ROOT.nullptr, "solid: p0+kx; dashed: kx", "")
        legend.Draw()

    canvas._legends = legends
    canvas.Update()
    return canvas


comparison_canvases = {group_name: draw_group_canvas(group_name) for group_name in OBSERVABLE_GROUPS}
comparison_canvases["vn_over_epsn"].Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_vn_over_epsn
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_vn_over_epsm


In [20]:
comparison_canvases["vn_over_epsm"].Draw()


In [21]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    csv_path = OUTPUT_DIR / "vn_epsn_pyroot_th2_fit_summary.csv"
    with csv_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0].keys()) if summary_rows else [])
        if summary_rows:
            writer.writeheader()
            writer.writerows(summary_rows)

    root_path = OUTPUT_DIR / "vn_epsn_pyroot_th2_fit.root"
    output_file = ROOT.TFile.Open(str(root_path), "RECREATE")
    for group_results in fit_results.values():
        for observable_results in group_results.values():
            for result in observable_results:
                result["profile"].Write()
                result["free_fn"].Write()
                result["origin_fn"].Write()
    for group_name, canvas in comparison_canvases.items():
        canvas.Write()
        canvas.SaveAs(str(OUTPUT_DIR / f"{group_name}.png"))
    output_file.Close()

    print(f"Saved summary CSV: {csv_path}")
    print(f"Saved ROOT profiles/fits/canvases: {root_path}")
else:
    print("SAVE_OUTPUTS is False; no files were written.")


Saved summary CSV: /Users/allenzhou/Research_software/Blast_wave/qa/vn_epsn_pyroot_th2_fit/vn_epsn_pyroot_th2_fit_summary.csv
Saved ROOT profiles/fits/canvases: /Users/allenzhou/Research_software/Blast_wave/qa/vn_epsn_pyroot_th2_fit/vn_epsn_pyroot_th2_fit.root


Info in <TCanvas::Print>: png file /Users/allenzhou/Research_software/Blast_wave/qa/vn_epsn_pyroot_th2_fit/vn_over_epsn.png has been created
Info in <TCanvas::Print>: png file /Users/allenzhou/Research_software/Blast_wave/qa/vn_epsn_pyroot_th2_fit/vn_over_epsm.png has been created
